<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/Tautologias_e_contradi%C3%A7%C3%B5es.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook: Variação lógica das tags e teste das expressões de tautologias e contradições

Este notebook gera valores lógicos aleatórios para todas as variáveis de processo mapeadas na **Linha de Envase e Tampamento** e avalia as expressões lógicas das regras de segurança, permissivos e validações formais.

A ideia é verificar, em um conjunto grande de combinações (exaustivo via `itertools`), se cada expressão é:
* **Tautologia:** sempre verdadeira (1)
* **Contradição:** sempre falsa (0)
* **Contingente:** depende dos valores das entradas (0 ou 1)

In [3]:
import itertools
import random
import pandas as pd

# Mapeamento completo das variáveis do processo de envase
variaveis = [
    'p_max1', 'p_min1', 'p_max2', 'p_min2',
    'q_max1', 'q_min1', 'q_max2', 'q_min2',
    'l_min', 'v_max', 'v_min', 'x_fc',
    'y_bomba', 'y_valv1', 'y_valv2', 'y_valv3', 'y_valv4', 'y_valv5', 'y_capp'
]

# Gera uma combinação aleatória dos valores lógicos das variáveis
def gerar_estado_aleatorio():
    return {v: random.choice([True, False]) for v in variaveis}

# Gera todas as combinações possíveis (exaustivo)
todas_as_variacoes = [
    dict(zip(variaveis, valores))
    for valores in itertools.product([False, True], repeat=len(variaveis))
]

print(f'Número total de variáveis: {len(variaveis)}')
print(f'Número total de combinações possíveis: {len(todas_as_variacoes)}')
print('\nExemplo de estado aleatório:')
print(gerar_estado_aleatorio())

Número total de variáveis: 19
Número total de combinações possíveis: 524288

Exemplo de estado aleatório:
{'p_max1': True, 'p_min1': True, 'p_max2': True, 'p_min2': True, 'q_max1': True, 'q_min1': True, 'q_max2': True, 'q_min2': False, 'l_min': False, 'v_max': True, 'v_min': True, 'x_fc': True, 'y_bomba': True, 'y_valv1': True, 'y_valv2': False, 'y_valv3': False, 'y_valv4': False, 'y_valv5': True, 'y_capp': True}


## Expressões lógicas do processo

As regras abaixo seguem as equações de intertravamento e permissivos modeladas no arquivo `03 - Tautologias e contradições.md`.

In [4]:
# A. Condição de Falha Crítica na Sucção/Alimentação
def F1(vars_):
    return vars_['p_min1'] or vars_['p_max1'] or vars_['q_max1']

# A. Intertrava de Trip: F1 -> (not y_bomba and not y_valv1)
def intertravamento_trip(vars_):
    return (not F1(vars_)) or (not vars_['y_bomba'] and not vars_['y_valv1'])

# B. Permissivo de Operação da Bomba BC1: y_bomba -> (y_valv1 and not p_min1 and not p_max1 and not p_max2)
def permissivo_bomba(vars_):
    p_bomba = vars_['y_valv1'] and (not vars_['p_min1']) and (not vars_['p_max1']) and (not vars_['p_max2'])
    return (not vars_['y_bomba']) or p_bomba

# C. Permissivo de Dosagem/Envase VS2: y_valv2 -> (pressão e vazão nominais e esteira OK)
def permissivo_dosagem(vars_):
    p_dose = (not vars_['p_min2']) and (not vars_['p_max2']) and \
             (not vars_['q_min2']) and (not vars_['q_max2']) and \
             (not vars_['v_max']) and (not vars_['v_min'])
    return (not vars_['y_valv2']) or p_dose

# D. Permissivo da Estação de Capping: (y_valv4 and y_capp) -> (l_min and not v_max and not v_min)
def permissivo_capping(vars_):
    p_cap = vars_['l_min'] and (not vars_['v_max']) and (not vars_['v_min'])
    return (not (vars_['y_valv4'] and vars_['y_capp'])) or p_cap

# Prova Formal de Segurança: Estado de Risco AND Regra de Intertrava
def risco_estado(vars_):
    return vars_['p_min1'] and vars_['y_bomba']

def regra_seguranca(vars_):
    # p_min1 -> not y_bomba  <=>  not p_min1 or not y_bomba
    return (not vars_['p_min1']) or (not vars_['y_bomba'])

def prova_contradicao(vars_):
    return risco_estado(vars_) and regra_seguranca(vars_)

# Exemplos clássicos de tautologia e contradição
def tautologia_classica(vars_):
    return vars_['p_max1'] or (not vars_['p_max1'])

def contradicao_classica(vars_):
    return vars_['p_max1'] and (not vars_['p_max1'])

# Dicionário com as expressões a serem avaliadas
expressoes = {
    'A. Intertrava de trip da alimentação': intertravamento_trip,
    'B. Permissivo de partida da bomba': permissivo_bomba,
    'C. Permissivo de dosagem/envase': permissivo_dosagem,
    'D. Permissivo da estação de capping': permissivo_capping,
    'Prova de segurança (p_min1 and y_bomba) AND (p_min1 -> not y_bomba)': prova_contradicao,
    'Tautologia clássica (p_max1 or not p_max1)': tautologia_classica,
    'Contradição clássica (p_max1 and not p_max1)': contradicao_classica
}

def classificar_expressao(fn):
    valores = [fn(v) for v in todas_as_variacoes]
    if all(valores):
        return 'Tautologia'
    elif not any(valores):
        return 'Contradição'
    return 'Contingente'

resultado = []
for nome, fn in expressoes.items():
    valores = [fn(v) for v in todas_as_variacoes]
    resultado.append({
        'Expressão': nome,
        'Verdadeiras': sum(valores),
        'Falsas': len(valores) - sum(valores),
        'Classificação': classificar_expressao(fn)
    })

print(pd.DataFrame(resultado).to_string(index=False))

                                                          Expressão  Verdadeiras  Falsas Classificação
                               A. Intertrava de trip da alimentação       180224  344064   Contingente
                                  B. Permissivo de partida da bomba       278528  245760   Contingente
                                    C. Permissivo de dosagem/envase       266240  258048   Contingente
                                D. Permissivo da estação de capping       409600  114688   Contingente
Prova de segurança (p_min1 and y_bomba) AND (p_min1 -> not y_bomba)            0  524288   Contradição
                         Tautologia clássica (p_max1 or not p_max1)       524288       0    Tautologia
                       Contradição clássica (p_max1 and not p_max1)            0  524288   Contradição


## Simulação aleatória de estados do processo

Abaixo são gerados alguns estados aleatórios para visualização do comportamento dinâmico das regras operacionais.

In [5]:
for i in range(5):
    estado = gerar_estado_aleatorio()
    print(f'\n--- Estado #{i+1} ---')
    print(f'Estado do Sistema: {estado}')
    print('F1 (Falha Crítica) =', F1(estado))
    print('Intertravamento de Trip =', intertravamento_trip(estado))
    print('Permissivo da Bomba =', permissivo_bomba(estado))
    print('Permissivo de Dosagem =', permissivo_dosagem(estado))
    print('Permissivo de Capping =', permissivo_capping(estado))
    print('Prova de Segurança (Contradição) =', prova_contradicao(estado))


--- Estado #1 ---
Estado do Sistema: {'p_max1': False, 'p_min1': False, 'p_max2': False, 'p_min2': False, 'q_max1': True, 'q_min1': True, 'q_max2': True, 'q_min2': False, 'l_min': False, 'v_max': False, 'v_min': False, 'x_fc': True, 'y_bomba': True, 'y_valv1': True, 'y_valv2': True, 'y_valv3': False, 'y_valv4': True, 'y_valv5': True, 'y_capp': True}
F1 (Falha Crítica) = True
Intertravamento de Trip = False
Permissivo da Bomba = True
Permissivo de Dosagem = False
Permissivo de Capping = False
Prova de Segurança (Contradição) = False

--- Estado #2 ---
Estado do Sistema: {'p_max1': True, 'p_min1': False, 'p_max2': False, 'p_min2': False, 'q_max1': True, 'q_min1': True, 'q_max2': True, 'q_min2': False, 'l_min': True, 'v_max': False, 'v_min': False, 'x_fc': False, 'y_bomba': False, 'y_valv1': True, 'y_valv2': False, 'y_valv3': False, 'y_valv4': True, 'y_valv5': True, 'y_capp': False}
F1 (Falha Crítica) = True
Intertravamento de Trip = False
Permissivo da Bomba = True
Permissivo de Dosagem

## Interpretação dos Resultados

* **Prova de Segurança:** O valor avalia sempre como `False` (0 ocorrências de `True`), confirmando que a conjunção do estado de risco $(p_{mín1} \land y_{bomba})$ com a regra de intertravamento $(p_{mín1} \rightarrow \neg y_{bomba})$ é uma **Contradição ($\bot$)**. Consequentemente, a sua negação é uma **Tautologia ($\top$)**, assegurando que a bomba nunca operará em subpressão destrutiva.
* **Regras de Processo e Permissivos:** As regras de intertravamento e permissivos operacionais são fórmulas **Contingentes**, pois dependem da combinação dinâmica dos sensores físicos da linha.
* **Validações Clássicas:** A expressão $p_{máx1} \lor \neg p_{máx1}$ resulta em **Tautologia** (100% verdadeira), e $p_{máx1} \land \neg p_{máx1}$ resulta em **Contradição** (0% verdadeira).